Task A) Design and implement a CNN model (with 4+ layers of convolutions) to classify multi category image datasets. Use the MNIST, Fashion MNIST, CIFAR-10 datasets. Set the No. of Epoch as 5, 10 and 20. Make the necessary changes whenever required. Record the accuracy corresponding to the number of epochs. Record the time required to run the program, using CPU as well as using GPU in Colab.

In [12]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import numpy as np
import time

# ----- CNN model – 5 convolutional layers -----
def create_model(input_shape):
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        Conv2D(32, (3,3), activation='relu'),
        MaxPooling2D(2,2),

        Conv2D(64, (3,3), activation='relu'),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),

        Conv2D(128, (3,3), activation='relu'),   # 5th conv layer
        MaxPooling2D(2,2),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ])
    return model

# ----- Load dataset and take a small subset -----
def load_dataset(name, max_train=3000, max_test=1000):
    if name == 'mnist':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
        x_train = np.expand_dims(x_train, -1)
        x_test  = np.expand_dims(x_test, -1)
    elif name == 'fashion_mnist':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
        x_train = np.expand_dims(x_train, -1)
        x_test  = np.expand_dims(x_test, -1)
    elif name == 'cifar10':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
        y_train = y_train.flatten()
        y_test  = y_test.flatten()
    else:
        raise ValueError("Unknown dataset")

    # Take only the first max_train and max_test samples
    x_train, y_train = x_train[:max_train], y_train[:max_train]
    x_test, y_test   = x_test[:max_test], y_test[:max_test]

    # Normalise
    x_train = x_train.astype('float32') / 255.0
    x_test  = x_test.astype('float32') / 255.0
    return (x_train, y_train), (x_test, y_test), x_train.shape[1:]

In [13]:
import subprocess, sys, json

# The subprocess script – same model, same subset sizes
cpu_script = """
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import numpy as np, time, json

def create_model(input_shape):
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        Conv2D(32, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ])
    return model

def load_dataset(name, max_train=3000, max_test=1000):
    if name == 'mnist':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
        x_train = np.expand_dims(x_train, -1)
        x_test  = np.expand_dims(x_test, -1)
    elif name == 'fashion_mnist':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
        x_train = np.expand_dims(x_train, -1)
        x_test  = np.expand_dims(x_test, -1)
    elif name == 'cifar10':
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
        y_train = y_train.flatten()
        y_test  = y_test.flatten()
    else:
        raise ValueError("Unknown dataset")
    x_train, y_train = x_train[:max_train], y_train[:max_train]
    x_test, y_test   = x_test[:max_test], y_test[:max_test]
    x_train = x_train.astype('float32') / 255.0
    x_test  = x_test.astype('float32') / 255.0
    return (x_train, y_train), (x_test, y_test), x_train.shape[1:]

datasets = ['mnist', 'fashion_mnist', 'cifar10']
epoch_list = [5, 10, 20]
results = []
for ds in datasets:
    (x_train, y_train), (x_test, y_test), input_shape = load_dataset(ds)
    for ep in epoch_list:
        model = create_model(input_shape)
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        start = time.time()
        model.fit(x_train, y_train, batch_size=128, epochs=ep, validation_split=0.1, verbose=0)
        elapsed = time.time() - start
        _, test_acc = model.evaluate(x_test, y_test, verbose=0)
        results.append({'dataset': ds, 'epochs': ep, 'accuracy': float(test_acc), 'time': elapsed})
print(json.dumps(results))
"""

print("=" * 50)
print("RUNNING ON CPU (subprocess, using 3000 train samples)")
print("=" * 50)

# No GPU for the subprocess
env = {**__import__('os').environ, 'CUDA_VISIBLE_DEVICES': '-1'}
proc = subprocess.run([sys.executable, '-c', cpu_script],
                      capture_output=True, text=True, env=env)

cpu_results_raw = json.loads(proc.stdout.strip())
cpu_results = [(r['dataset'], r['epochs'], r['accuracy'], r['time']) for r in cpu_results_raw]

for ds, ep, acc, t in cpu_results:
    print(f"Epochs: {ep:2d} | Test Accuracy: {acc:.4f} | Time: {t:.2f} s")
print("\nCPU training complete.")

RUNNING ON CPU (subprocess, using 3000 train samples)
Epochs:  5 | Test Accuracy: 0.9300 | Time: 25.97 s
Epochs: 10 | Test Accuracy: 0.9570 | Time: 47.13 s
Epochs: 20 | Test Accuracy: 0.9680 | Time: 92.73 s
Epochs:  5 | Test Accuracy: 0.7340 | Time: 25.82 s
Epochs: 10 | Test Accuracy: 0.7790 | Time: 47.13 s
Epochs: 20 | Test Accuracy: 0.8360 | Time: 91.91 s
Epochs:  5 | Test Accuracy: 0.2790 | Time: 45.22 s
Epochs: 10 | Test Accuracy: 0.3830 | Time: 90.70 s
Epochs: 20 | Test Accuracy: 0.4620 | Time: 176.06 s

CPU training complete.


In [14]:
# ----- Training on GPU (same subset) -----
print("=" * 50)
print("RUNNING ON GPU (using 3000 train samples)")
print("=" * 50)

datasets = ['mnist', 'fashion_mnist', 'cifar10']
epoch_list = [5, 10, 20]
gpu_results = []   # (dataset, epochs, accuracy, time)

for ds in datasets:
    # use exactly the same subset sizes as CPU
    (x_train, y_train), (x_test, y_test), input_shape = load_dataset(ds, max_train=3000, max_test=1000)
    print(f"\n--- Dataset: {ds} ---")
    for ep in epoch_list:
        model = create_model(input_shape)
        model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])
        start = time.time()
        model.fit(x_train, y_train, batch_size=128, epochs=ep,
                  validation_split=0.1, verbose=2)
        train_time = time.time() - start
        _, test_acc = model.evaluate(x_test, y_test, verbose=0)
        gpu_results.append((ds, ep, test_acc, train_time))
        print(f"Epochs: {ep:2d} | Test Accuracy: {test_acc:.4f} | Time: {train_time:.2f} s")

print("\nGPU training complete.")

RUNNING ON GPU (using 3000 train samples)

--- Dataset: mnist ---
Epoch 1/5
22/22 - 9s - 389ms/step - accuracy: 0.2774 - loss: 2.1011 - val_accuracy: 0.7067 - val_loss: 1.0906
Epoch 2/5
22/22 - 0s - 10ms/step - accuracy: 0.6556 - loss: 1.0192 - val_accuracy: 0.8333 - val_loss: 0.5701
Epoch 3/5
22/22 - 0s - 10ms/step - accuracy: 0.8330 - loss: 0.5279 - val_accuracy: 0.8967 - val_loss: 0.3689
Epoch 4/5
22/22 - 0s - 10ms/step - accuracy: 0.8985 - loss: 0.3401 - val_accuracy: 0.9300 - val_loss: 0.2745
Epoch 5/5
22/22 - 0s - 10ms/step - accuracy: 0.9052 - loss: 0.3213 - val_accuracy: 0.9167 - val_loss: 0.2685
Epochs:  5 | Test Accuracy: 0.9200 | Time: 9.51 s
Epoch 1/10
22/22 - 8s - 352ms/step - accuracy: 0.3067 - loss: 1.9380 - val_accuracy: 0.6533 - val_loss: 1.0566
Epoch 2/10
22/22 - 0s - 13ms/step - accuracy: 0.6852 - loss: 0.9243 - val_accuracy: 0.8633 - val_loss: 0.4872
Epoch 3/10
22/22 - 0s - 12ms/step - accuracy: 0.8307 - loss: 0.5359 - val_accuracy: 0.9100 - val_loss: 0.3053
Epoch 4

In [15]:
# ----- Side‑by‑side comparison -----
print("\n========== GPU vs CPU (3000 train samples) ==========")
print(f"{'Dataset':<12} {'Epochs':<7} {'GPU Acc':<9} {'GPU Time':<10} {'CPU Acc':<9} {'CPU Time':<10}")
print("-" * 60)

for g, c in zip(gpu_results, cpu_results):
    ds, ep, acc_g, time_g = g
    _, _, acc_c, time_c = c
    print(f"{ds:<12} {ep:<7} {acc_g:<9.4f} {time_g:<10.2f} {acc_c:<9.4f} {time_c:<10.2f}")


========== GPU vs CPU (3000 train samples) ==========
Dataset      Epochs  GPU Acc   GPU Time   CPU Acc   CPU Time  
------------------------------------------------------------
mnist        5       0.9200    9.51       0.9300    25.97     
mnist        10      0.9450    9.95       0.9570    47.13     
mnist        20      0.9440    12.26      0.9680    92.73     
fashion_mnist 5       0.7310    8.62       0.7340    25.82     
fashion_mnist 10      0.7650    10.07      0.7790    47.13     
fashion_mnist 20      0.8130    12.31      0.8360    91.91     
cifar10      5       0.2560    10.73      0.2790    45.22     
cifar10      10      0.3560    10.50      0.3830    90.70     
cifar10      20      0.4470    13.31      0.4620    176.06    
